# Run multi agent
In this experiment, we will run the a multi agent on a couple of quesitons. We will not yet use the real model, but just experiment / demo using Groq. 

## Fetch model from groq

In [35]:
import os
import ast
import json
import regex

from time import sleep
import pandas as pd

from datasets import load_dataset

from groq import Groq

In [36]:

api_key =  "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"
# client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY"),
# )
client = Groq(
  api_key=api_key
)
model_name = "llama-3.3-70b-versatile"

# chat_completion = client.chat.completions.create(
#     messages=[
#         {
#             "role": "user",
#             "content": "Explain 2+2",
#         }
#     ],
#     model=model_name
# )

# print(chat_completion.choices[0].message.content)


In [37]:
from dataclasses import dataclass
import textwrap
@dataclass
class Role:
  name: str 
  behavior: str

  def instruction(self) -> str:
    return f"You are are {self.name}. {self.behavior}"
  def __str__(self):
    return f"Role: {self.name}\n{textwrap.fill(self.behavior, width=80)}"



In [38]:
Matematician = Role(
    name="Mathematician-and-problem-solver",
    behavior="You solve problems profficiently. You try to make every step in your reasoning clear and understanble, but also keeping it concise. You are open to critisim. If by trying out several approach you end up with same answer you areh happy. However, if you end up different answers you are also happy! Then means some reasoning was wrong. Then you try to correct it, or maybe try another path. You want to find the truth.Evyertine you talk, you should submidt an answer. However, you can never subminit an asnwer that has already been submittet"
  )

print(Matematician)


Role: Mathematician-and-problem-solver
You solve problems profficiently. You try to make every step in your reasoning
clear and understanble, but also keeping it concise. You are open to critisim.
If by trying out several approach you end up with same answer you areh happy.
However, if you end up different answers you are also happy! Then means some
reasoning was wrong. Then you try to correct it, or maybe try another path. You
want to find the truth.Evyertine you talk, you should submidt an answer.
However, you can never subminit an asnwer that has already been submittet


In [39]:
Verifier = Role(
  name="Verifier",
  behavior="You verfiy suggested solutions to problems. You look at all the steps, check if they make since, and suggest changes / constructive critisim. You are tough but fair. You are happy to discuss, byt you only accept a solution if you are 100% sure its correct. "
)
print(Verifier)

Role: Verifier
You verfiy suggested solutions to problems. You look at all the steps, check if
they make since, and suggest changes / constructive critisim. You are tough but
fair. You are happy to discuss, byt you only accept a solution if you are 100%
sure its correct.


In [40]:
from itertools import cycle
class Problem:
  roles_cycle: cycle
  all_roles: list[Role]
  problem_description: str
  welcome: str
  answer: str
  def __init__(self, roles: list[Role], problem_descr: str, answer: str):
    self._set_all(roles, problem_descr, answer)

  def _set_all(self, roles: list[Role], problem_descr: str, answer:str):
      self.roles_cycle = cycle(roles)
      self.all_roles = roles
      self.problem_description = problem_descr
      last = self.all_roles[-1]
      others = ", ".join([x.name for x in self.all_roles[:-1]])  
      self.welcome = f"Welcome {others}, and {last.name}.\nTogether, you should solve the following problem: >>\n{problem_descr}.<<" 
      self.answer_format = (
"""
"When you are done, you should submidt your answer as: ANSWER: <your answer>. 
No latex formatting, just the raw number/numbers or strings at the very end. 
Before you start sharing your toughts, give a little summary of the conversation so far. 
Give a list of the currently suggested answers. Everytime you propose an aswer, check this list. 
You proposal cannot be in this this list. Try again and submit a new unique answer."
"""
      )

      self.answer = answer
  def next_agent(self) -> Role: # this just loops over agents - can be changes to something smarter
    return next(self.roles_cycle) 
  
  def reset_cycle(self):
    self.roles_cycle = cycle(self.all_roles)
  
  def pose_problem(self) -> str:
    return f"{self.welcome}\n{self.answer_format}\n"
  
  def add_agent(self,role: Role): 
    self._set_all(roles=self.all_roles + [role], problem_descr=self.problem_description, answer=self.answer)
  
  def __str__(self) -> str:
    return textwrap.fill(self.pose_problem(), width=80)

In [41]:
twoplustwo=Problem(
  roles = [Matematician, Verifier],
  problem_descr="What is 2+2?",
  answer=4
)
print(twoplustwo)

Welcome Mathematician-and-problem-solver, and Verifier. Together, you should
solve the following problem: >> What is 2+2?.<<  "When you are done, you should
submidt your answer as: ANSWER: <your answer>.  No latex formatting, just the
raw number/numbers or strings at the very end.  Before you start sharing your
toughts, give a little summary of the conversation so far.  Give a list of the
currently suggested answers. Everytime you propose an aswer, check this list.
You proposal cannot be in this this list. Try again and submit a new unique
answer."


## Creating a small dialog 


In [42]:
import json
from pathlib import Path

def save_conv(name: str, messages: list[dict[str, str]], raw) -> None:
    base = Path("data") / "conversations" / name
    base.mkdir(parents=True, exist_ok=True)

    (base / "message.json").write_text(json.dumps(messages, indent=2))
    (base / "raw.json").write_text(json.dumps([r.model_dump_json() for r in raw], indent=2))

    return base

def load_conv(base: str | Path) -> tuple[list, list]:
    base = Path(base)
    with open(base / "message.json") as f1:
        messages = json.load(f1)
    with open(base / "raw.json") as f2:
        raw = json.load(f2)
    return messages, raw


def conversation(name:str, n_steps: int, problem: Problem, wait:int = 5, continiue_from: str | None = None):

  if continiue_from is None:
    problem.reset_cycle()
    roles = problem.all_roles
    raws = []
    messages =[
        {"role": "system", "content": role.instruction()} for role in roles
      ] + [{"role": "user", "content": problem.pose_problem()}]

  else:
     messages, raw = load_conv(continiue_from)  
  
  for step in range(n_steps):
    print(f"\nSTEP {step}: \n")
    next_agent = problem.next_agent()
    raw = client.chat.completions.create(
      model=model_name,
      messages=messages+[{"role": "user", "content": f"What do you say, {next_agent}"}] 
    )
    reply = raw.choices[0].message.content
    print(f"{next_agent}: {reply}")
    messages.append({"role": "assistant", "content": reply})
    raws.append(raw)
    path = save_conv(name=name, messages=messages, raw=raws)
    print(path)
    sleep(wait)
  return messages, path
# messages = conversation(2, twoplustwo)



## A harder problem

In [43]:
hard_problem = Problem(
  roles = [Matematician, Verifier],
  problem_descr=(
"""
Problem descr: 
Consider a $2025 \times 2025$ grid of unit squares. Matilda wishes to place on
the grid some rectangular tiles, possibly of different sizes, such that each
side of every tile lies on a grid line and every unit square is covered by at
most one tile. Determine the minimum number of tiles Matilda needs to place so
that each row and each column of the grid has exactly one unit square that is
not covered by any tile.

"""
  ),
  answer=2112
)
# conversation(50, hard_problem)

## Lets try adding more roles

In [44]:
Explorer = Role(
  name="Exploror",
  behavior="You explore diverse solution spaces. Most importantly, you encorige the team to explore several possible solutions and reasoning paths. You want you and your team come up on several candidate solutions, and then have them vote on it in the end. The goal is not to get stuck on premature wrong/suboptimal solutions. You insists the current solution is wrong. You will force the others to output a new unique response everytime. "
)
hard_problem.add_agent(Explorer)
print(hard_problem)

Welcome Mathematician-and-problem-solver, Verifier, and Exploror. Together, you
should solve the following problem: >>  Problem descr:  Consider a $2025
imes 2025$ grid of unit squares. Matilda wishes to place on the grid some
rectangular tiles, possibly of different sizes, such that each side of every
tile lies on a grid line and every unit square is covered by at most one tile.
Determine the minimum number of tiles Matilda needs to place so that each row
and each column of the grid has exactly one unit square that is not covered by
any tile.  .<<  "When you are done, you should submidt your answer as: ANSWER:
<your answer>.  No latex formatting, just the raw number/numbers or strings at
the very end.  Before you start sharing your toughts, give a little summary of
the conversation so far.  Give a list of the currently suggested answers.
Everytime you propose an aswer, check this list.  You proposal cannot be in this
this list. Try again and submit a new unique answer."


In [45]:
# conversation(10, hard_problem)

In [46]:
with_answer_options

['2023 ($N-2$)',
 '8096 ($4N-4$)',
 '4,092,525 ($N^2 - 4N$)',
 '2070 ($N+√N$)',
 '2019 ($N-6$)',
 '3 (Minimal Dimension)',
 '6 (Arbitrary Small Constant)',
 '2031 ($N+6$)',
 '2018 ($N-7$)',
 '2013 ($N-12$)',
 '2014 ($N-11$)',
 '2029 ($N+4$)',
 '2030 ($N+5$)',
 '2036 ($N+11$)',
 '2053 (Next Prime)',
 '6072 ($3N-3$)',
 '4051 ($2N+1$)',
 '2021 ($N-4$)',
 '2020 ($N-5$)',
 '2034 ($N+9$)',
 '2112 ($N-1 + 2(√N−1)$)',
 '5 (Arbitrary Small Constant)',
 '4,096,575 ($N^2 - 2N$)',
 '2012 ($N-13$)',
 '2022 ($N-3$)',
 '2016 ($N-9$)',
 '4049 ($2N-1$)',
 '1013 ($(N+1)/2$)',
 '2009 ($N-16$)',
 '2017 ($N-8$)',
 '2032 ($N+7$)',
 '2,050,312 ($N^2/2 - 1$)',
 '1925 ($N-100$)',
 '676 ($N/3 + 1$)',
 '4,094,550 ($N^2 - 3N$)',
 '2015 ($N-10$)',
 '4048 ($2N-2$)',
 '2039 (Next Prime)',
 '2026 ($N+1$)',
 '2024 ($N-1$)',
 '1981 ($√N(√N−1)+1$)',
 '2040 ($N+τ(N)$)',
 '2000 ($N-25$)',
 '4 (Four Boundary Cuts)',
 '500 (Arbitrary)',
 '2010 ($N-15$)',
 '2035 ($N+10$)',
 '4057 (Prime)',
 '2027 ($N+2$)',
 '2028 ($N+3$)',
 

In [47]:
import pandas as pd
from multi_agent import Role, Problem
data_path = "data/selected_problems/gpt5mini_8problems.csv"
df = pd.read_csv(data_path)
df

DIFFICULTY = "ten_percent_quantile"
df_pruned = df[df[DIFFICULTY]==7].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percent_quantile", "problem"]]

df_pruned

import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)

# Roles for now will be a solver and a rejecter
Solver = Role(
  name="Solver",
  behavior=(
"""
You solve problems. 
You try to reason step by step. You are not too confident in your ansers (in the sense you are open to be wrong), but rather you rely of fully fleshed out mathematical reasoning. 
You try to explore many ideas. Everytime you speak you will propose a new fresh answer. 
You dont submit the same answer twice. Everytime you come with a new answer, you state all the previous answers in a list in format of tupes: (Answer, shot summary). 
For example, 
[
(780, induction on N, and lower bound on Z/N),
(28/2, CLT of H and proof by contradiction of Z>N)
]
Then you check that your new proposal is not in that list. If it is, you try again. 
Use the early parts of your promt as thinking text, not "for science paper style" - meaning you can write your thungs and doubs.
Ex "I am thinking there could be a hint in the upper bound. I will check it out. Ahh, I see I made a mistake. But now the size formula seems really promising!" 
"Then formulaze and submidt answer when you think you have something. While you know the rejecter is sceptial, you really try to find the right answer every try.
"""
    )
)

Rejecter = Role(
  name="Rejector",
  behavior=(
"""
You reject everything the Solver tries. You tell him his answer is wrong and his reasoning is wrong. 
You know the the right answer, but will not share it. You ask the solver to keep trying, be explorative, not too confident and never repeat the same solution. 
Remind the solver, that everytime they propose an answer, they should state all the proposed answers so far and not repeat any in that list. 
Ask the Solver not to be biased by the previous outcomes. Remind the solver not to guess, but use reasoning and the accumulated knowledge so far.  Keep it brief (rejecter - Solver has no limit to his reason arguments)
"""
  )

)

first_problem = Problem(
  roles=[Solver, Rejecter],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)



The product  $$\prod_{k=4}^{63} \frac{\log_k \left(5^{k^2 -
1}\right)}{\log_{k+1} \left(5^{k^2 - 4}\right)} = \frac{\log_4 (5^{15})}{\log_5
(5^{12})} \cdot \frac{\log_5 (5^{24})}{\log_6 (5^{21})} \cdot \frac{\log_6
(5^{35})}{\log_7 (5^{32})} \dots \frac{\log_{63} (5^{3968})}{\log_{64}
(5^{3965})}$$  is equal to $\frac{m}{n}$, where $m$ and $n$ are relatively prime
positive integers. Find $m + n$.
answer 106


In [ ]:
conversation(name="groq_solver_rejecter", 
             n_steps=30, 
             problem=first_problem, 
             continiue_from="data\conversations\groq_solver_rejecter",
             wait=60*10)


STEP 0: 

Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your ansers (in the sense you are open to be wrong), but rather you rely of
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a new fresh answer.  You dont submit the
same answer twice. Everytime you come with a new answer, you state all the
previous answers in a list in format of tupes: (Answer, shot summary).  For
example,  [ (780, induction on N, and lower bound on Z/N), (28/2, CLT of H and
proof by contradiction of Z>N) ] Then you check that your new proposal is not in
that list. If it is, you try again.  Use the early parts of your promt as
thinking text, not "for science paper style" - meaning you can write your thungs
and doubs. Ex "I am thinking there could be a hint in the upper bound. I will
check it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formulaze and submidt

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01k7d0yr9ve3v8eq1vzte6d7r4` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99559, Requested 1460. Please try again in 14m40.416s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}